In [1]:
from pathlib import Path
import sys

project_dir = Path.cwd()

required_files = [
    "data/evaluation/development_v1.json",
    "data/access_control/policy_v1.json",
    "config/incident_lab/retrieval_v1.json",
]

missing = [
    name for name in required_files
    if not (project_dir / name).is_file()
]

if missing:
    raise RuntimeError(f"Missing project files: {missing}")

print("Project folder:", project_dir)
print("Python executable:", sys.executable)
print("Integration prerequisites found:", len(required_files))

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Integration prerequisites found: 3


In [2]:
import workplace_pipeline as pipeline

pipeline.validate_top_k(2)

tokens = pipeline.validate_question(
    "How many annual leave days do full-time employees receive?"
)

policy_file = pipeline.resolve_project_path(
    project_dir,
    "data/access_control/policy_v1.json",
)

policy = pipeline.read_json_object(policy_file)

print("Module loaded:", pipeline.__file__)
print("Configuration accepted: top_k=2")
print("Searchable query tokens:", len(tokens))
print("Policy roles:", ", ".join(policy["roles"]))

Module loaded: C:\Users\Khaleel\workplace-ai-operations-lab\workplace_pipeline.py
Configuration accepted: top_k=2
Searchable query tokens: 8
Policy roles: employee, hr_admin


In [3]:
import importlib

# Reload so this kernel sees the saved module changes.
pipeline = importlib.reload(pipeline)

for role, expected_count in [("employee", 9), ("hr_admin", 11)]:
    role_passages, read_ids = pipeline.load_authorized_passages(
        project_dir,
        "data/access_control/policy_v1.json",
        role,
    )

    assert len(role_passages) == expected_count

    if role == "employee":
        assert "HR-RESTRICTED-001" not in read_ids
        assert all(
            item["document_id"] != "HR-RESTRICTED-001"
            for item in role_passages
        )
    else:
        assert "HR-RESTRICTED-001" in read_ids

    print(
        f"PASS: {role} | "
        f"documents={len(read_ids)} | passages={len(role_passages)}"
    )

try:
    pipeline.load_authorized_passages(
        project_dir,
        "data/access_control/policy_v1.json",
        "unknown_role",
    )
except pipeline.AccessDeniedError:
    print("PASS: Unknown role denied.")
else:
    raise AssertionError("Unknown role was not denied.")

PASS: employee | documents=3 | passages=9
PASS: hr_admin | documents=4 | passages=11
PASS: Unknown role denied.


In [4]:
import json

integrated_config_dir = project_dir / "config" / "integrated"
integrated_config_dir.mkdir(parents=True, exist_ok=True)

integrated_config = {
    "schema_version": "1.0",
    "version": "1",
    "policy_path": "data/access_control/policy_v1.json",
    "top_k": 2,
}

integrated_config_path = integrated_config_dir / "pipeline_v1.json"
config_text = json.dumps(integrated_config, indent=2) + "\n"

if integrated_config_path.exists():
    if integrated_config_path.read_text(encoding="utf-8") != config_text:
        raise RuntimeError("Existing integrated configuration differs.")
    print("Integrated configuration already matches.")
else:
    integrated_config_path.write_text(config_text, encoding="utf-8")
    print("Created:", integrated_config_path)

# Check the saved configuration through our module.
saved_config = pipeline.read_json_object(integrated_config_path)
pipeline.validate_top_k(saved_config["top_k"])

configured_policy_path = pipeline.resolve_project_path(
    project_dir,
    saved_config["policy_path"],
)

configured_policy = pipeline.read_json_object(configured_policy_path)

print("Configured top_k:", saved_config["top_k"])
print("Policy documents:", len(configured_policy["documents"]))
print("Roles:", ", ".join(configured_policy["roles"]))

Integrated configuration already matches.
Configured top_k: 2
Policy documents: 4
Roles: employee, hr_admin


In [5]:
import importlib
from uuid import uuid4

pipeline = importlib.reload(pipeline)

integrated_log_dir = project_dir / "evidence" / "integrated_logs"
integrated_log_dir.mkdir(parents=True, exist_ok=True)
integrated_log_path = integrated_log_dir / f"requests_{uuid4().hex}.jsonl"

integrated_response = pipeline.retrieve_request(
    project_root=project_dir,
    config_relative_path="config/integrated/pipeline_v1.json",
    question="How many annual leave days do full-time employees receive?",
    role="employee",
    log_path=integrated_log_path,
)

assert integrated_response["status"] == "ok"
assert integrated_response["results"][0]["passage_id"] == "HR-001:1"

print("Status:", integrated_response["status"])
print("Configuration version:", integrated_response["config_version"])
print("Top passage:", integrated_response["results"][0]["passage_id"])
print("Log file:", integrated_log_path)

Status: ok
Configuration version: 1
Top passage: HR-001:1
Log file: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\integrated_logs\requests_f71c9989a1b042859cad75e8edc75189.jsonl


In [6]:
restricted_question = (
    "What is the reference code for the internal HR review procedure?"
)

integration_checks = [
    ("Employee restricted query", restricted_question, "employee",
     "config/integrated/pipeline_v1.json", "no_match", None),
    ("HR administrator restricted query", restricted_question, "hr_admin",
     "config/integrated/pipeline_v1.json", "ok", None),
    ("Unknown role", restricted_question, "unknown_role",
     "config/integrated/pipeline_v1.json", "error", "ACCESS_DENIED"),
    ("Empty query", "   ", "employee",
     "config/integrated/pipeline_v1.json", "error", "INVALID_QUERY"),
    ("Absent topic", "What is the cafeteria breakfast menu?", "employee",
     "config/integrated/pipeline_v1.json", "no_match", None),
    ("Missing configuration", restricted_question, "employee",
     f"config/integrated/missing_{uuid4().hex}.json",
     "error", "INVALID_CONFIGURATION"),
]

integration_responses = [integrated_response]

for name, question, role, config_path, expected_status, expected_error in (
    integration_checks
):
    result = pipeline.retrieve_request(
        project_root=project_dir,
        config_relative_path=config_path,
        question=question,
        role=role,
        log_path=integrated_log_path,
    )

    assert result["status"] == expected_status, (name, result)
    assert result["error_code"] == expected_error, (name, result)

    if expected_status != "ok":
        assert result["results"] == [], name
        assert result["result_count"] == 0, name

    if role == "employee":
        assert all(
            item["document_id"] != "HR-RESTRICTED-001"
            and "HR-REVIEW-742" not in item["text"]
            for item in result["results"]
        ), name

    if name == "HR administrator restricted query":
        assert result["results"][0]["passage_id"] == "HR-RESTRICTED-001:1"
        assert "HR-REVIEW-742" in result["results"][0]["text"]

    integration_responses.append(result)
    print(
        f"PASS: {name} | status={result['status']} | "
        f"error={result['error_code']}"
    )

events = [
    json.loads(line)
    for line in integrated_log_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

for result in integration_responses:
    matching = [
        event for event in events
        if event["request_id"] == result["request_id"]
    ]
    assert len(matching) == 1
    assert matching[0] == {
        key: value for key, value in result.items()
        if key != "results"
    }

print("\nVerified request/log pairs:", len(integration_responses))

PASS: Employee restricted query | status=no_match | error=None
PASS: HR administrator restricted query | status=ok | error=None
PASS: Unknown role | status=error | error=ACCESS_DENIED
PASS: Empty query | status=error | error=INVALID_QUERY
PASS: Absent topic | status=no_match | error=None
PASS: Missing configuration | status=error | error=INVALID_CONFIGURATION

Verified request/log pairs: 7


In [7]:
from pathlib import Path
from unittest.mock import patch

original_read_text = Path.read_text

restricted_file = (
    project_dir / "data/access_control/restricted/hr_review_policy.txt"
).resolve()

allowed_file = (
    project_dir / "data/synthetic/leave_policy.txt"
).resolve()

attempted_reads = []


def block_restricted_read(path, *args, **kwargs):
    resolved = path.resolve()
    attempted_reads.append(resolved)

    if resolved == restricted_file:
        raise AssertionError("Restricted content read attempted.")

    return original_read_text(path, *args, **kwargs)


with patch.object(Path, "read_text", block_restricted_read):
    guarded_response = pipeline.retrieve_request(
        project_root=project_dir,
        config_relative_path="config/integrated/pipeline_v1.json",
        question=restricted_question,
        role="employee",
        log_path=integrated_log_path,
    )

expected_reads = {
    integrated_config_path.resolve(),
    policy_file.resolve(),
    (project_dir / "data/synthetic/leave_policy.txt").resolve(),
    (project_dir / "data/synthetic/remote_work_policy.txt").resolve(),
    (project_dir / "data/synthetic/it_support_policy.txt").resolve(),
}

assert set(attempted_reads) == expected_reads
assert len(attempted_reads) == 5
assert guarded_response["status"] == "no_match"
assert restricted_file not in attempted_reads

print("PASS: Integrated employee path read only configuration, policy,")
print("      and three allowed documents; no restricted content read.")


def simulate_unavailable_document(path, *args, **kwargs):
    if path.resolve() == allowed_file:
        raise FileNotFoundError("Simulated authorized-document failure.")

    return original_read_text(path, *args, **kwargs)


with patch.object(Path, "read_text", simulate_unavailable_document):
    unavailable_response = pipeline.retrieve_request(
        project_root=project_dir,
        config_relative_path="config/integrated/pipeline_v1.json",
        question="How many annual leave days are available?",
        role="employee",
        log_path=integrated_log_path,
    )

assert unavailable_response["status"] == "error"
assert unavailable_response["error_code"] == "SOURCE_UNAVAILABLE_OR_INVALID"
assert unavailable_response["results"] == []
assert unavailable_response["result_count"] == 0

integration_responses.extend([
    guarded_response,
    unavailable_response,
])

print("PASS: Unavailable authorized document produces an explicit error.")
print("Verified response cases so far:", len(integration_responses))

PASS: Integrated employee path read only configuration, policy,
      and three allowed documents; no restricted content read.
PASS: Unavailable authorized document produces an explicit error.
Verified response cases so far: 9


In [8]:
from tempfile import TemporaryDirectory
from collections import Counter

# Keep the temporary configuration inside the project root.
# The directory is removed automatically after this test.
with TemporaryDirectory(
    prefix="integration_test_",
    dir=project_dir,
) as temporary_dir:
    invalid_config_path = Path(temporary_dir) / "invalid_config.json"

    invalid_config = {
        **integrated_config,
        "version": "invalid-test",
        "top_k": 0,
    }

    invalid_config_path.write_text(
        json.dumps(invalid_config, indent=2),
        encoding="utf-8",
    )

    invalid_response = pipeline.retrieve_request(
        project_root=project_dir,
        config_relative_path=invalid_config_path.relative_to(
            project_dir
        ).as_posix(),
        question="How many annual leave days are available?",
        role="employee",
        log_path=integrated_log_path,
    )

assert invalid_response["status"] == "error"
assert invalid_response["error_code"] == "INVALID_CONFIGURATION"
assert invalid_response["results"] == []
assert invalid_response["result_count"] == 0

integration_responses.append(invalid_response)
print("PASS: Invalid integrated configuration rejected.")

# Verify every response against its logged completion event.
events = [
    json.loads(line)
    for line in integrated_log_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

expected_fields = {
    "timestamp_utc", "request_id", "event", "config_version",
    "status", "error_code", "result_count", "elapsed_ms",
}

for event in events:
    assert set(event) == expected_fields
    assert event["elapsed_ms"] >= 0

request_ids = [event["request_id"] for event in events]
assert len(request_ids) == len(set(request_ids))

for result in integration_responses:
    matching = [
        event for event in events
        if event["request_id"] == result["request_id"]
    ]
    assert len(matching) == 1
    assert matching[0] == {
        key: value for key, value in result.items()
        if key != "results"
    }

counts = Counter(event["status"] for event in events)

print("Verified request/log pairs:", len(integration_responses))
print("Logged requests:", len(events))

for status in ["ok", "no_match", "error"]:
    print(f"{status}: {counts[status]}")

print("Log schema and unique request IDs verified.")

PASS: Invalid integrated configuration rejected.
Verified request/log pairs: 10
Logged requests: 10
ok: 2
no_match: 3
error: 5
Log schema and unique request IDs verified.
